# Embeddings, Vectors & Vector Databases
### Module 4 — RAG Foundations (part 1 of 2)

**Goal:** build the engine under RAG, in the order Omar Santos teaches it — embeddings and embedding models, real cybersecurity data, chunking, and a **vector database (ChromaDB)** you install and operate yourself. The companion notebook `rag_example.ipynb` then assembles the full RAG pipeline.

**No TF-IDF and no LangChain** here — just plain Python, `sentence-transformers`, and `chromadb`, so every step is visible. LangChain is the next module.

> **Sources & credit:** Omar Santos — LinkedIn Learning course *RAG, AI Apps, and AI Agents for Cybersecurity and Networking*; book *Agentic AI for Cybersecurity* (Ch. 2); repo https://github.com/santosomar/AI-agents-for-cybersecurity. CVE dataset and `ssrf.txt` are from that repo.

> **Reproducibility:** runs offline (local embedding model + local Chroma). The OpenAI cell is optional and guarded on a key.

## 🛠️ Setup

In [1]:
# If needed:  !pip install -q numpy pandas scikit-learn matplotlib sentence-transformers chromadb
import numpy as np
import pandas as pd
try:
    import matplotlib.pyplot as plt
except Exception as e:                 # some locked-down machines block matplotlib's backend
    plt = None
    print("matplotlib unavailable here:", e)

np.set_printoptions(precision=3, suppress=True)
print("Setup ready.")

Setup ready.


# 🧭 1 — What Is RAG (and Why Embeddings)?

- LLMs have **static, generic** knowledge and can hallucinate
- **RAG** = Retrieve relevant docs → Augment the prompt → Generate
- Two phases: **ingestion** (build the knowledge base) + **query** (answer)
- Retrieval works by **meaning** — which needs **embeddings**
- This notebook builds the retrieval engine; the next does full RAG

> ### 🎤 Instructor Script
>
> Let's frame the whole module. A language model only knows what it was trained on — static, generic, and prone to making things up. Retrieval-Augmented Generation fixes that with three steps: retrieve relevant documents, augment the prompt with them, and generate an answer grounded in those documents. There are two phases: ingestion, where we chunk and embed our documents into a knowledge base, and query, where we embed the question, find the most relevant chunks, and feed them to the model. Everything hinges on retrieving by meaning, and that's what embeddings give us. So we'll build the engine — embeddings, then a vector database — and the next notebook assembles the full pipeline.

In [2]:
# RAG happens in two phases. Here are the steps, in plain English.
print("INGESTION  (done once, to build the knowledge base):")
print("   1. load your documents")
print("   2. split them into small chunks")
print("   3. turn each chunk into an embedding (a list of numbers)")
print("   4. save those vectors in a vector database")
print()
print("QUERY  (done every time someone asks a question):")
print("   1. turn the question into an embedding")
print("   2. search the database for the closest chunks")
print("   3. add those chunks to the prompt")
print("   4. the LLM writes an answer using them")

INGESTION  (done once, to build the knowledge base):
   1. load your documents
   2. split them into small chunks
   3. turn each chunk into an embedding (a list of numbers)
   4. save those vectors in a vector database

QUERY  (done every time someone asks a question):
   1. turn the question into an embedding
   2. search the database for the closest chunks
   3. add those chunks to the prompt
   4. the LLM writes an answer using them


# 🧠 2 — Embeddings & Embedding Models

- An **embedding** = a dense vector that captures *meaning*
- Similar meaning → vectors point in similar directions
- An **embedding model** turns text → vector (same space for all text)
- Local: `sentence-transformers` (MiniLM, 384-dim); OpenAI: 1536/3072-dim
- Compare models on the **MTEB** leaderboard (huggingface.co/mteb)

> ### 🎤 Instructor Script
>
> An embedding is just a list of numbers — a dense vector — that places a piece of text at a point in meaning-space, so things that mean similar things land near each other. The thing that produces them is an embedding model, and the rule is to embed everything with the same model so the vectors are comparable. We use a small local model, MiniLM, which is free and offline and gives 384 numbers per text; OpenAI's models give 1536 or 3072. Watch the output: we embed a few security phrases, print part of a vector, and then a similarity table — you'll see 'SQL injection' and 'command injection' score high together while 'weak TLS cipher' sits apart, even though they share no keywords.

In [3]:
from sentence_transformers import SentenceTransformer
import numpy as np

# a small, free embedding model that runs on your own computer
model = SentenceTransformer("all-MiniLM-L6-v2")

phrases = ["SQL injection in the login form",
           "command injection via crafted input",
           "weak TLS cipher configuration"]

# turn each phrase into a vector (a list of numbers that captures its meaning)
vectors = model.encode(phrases)
print("Each phrase became a list of", len(vectors[0]), "numbers.")
print("First phrase, first 8 numbers:", np.round(vectors[0][:8], 3))


# a small helper: how similar are two vectors?  1.0 = same meaning, 0 = unrelated
def similarity(vector_a, vector_b):
    dot = np.dot(vector_a, vector_b)
    sizes = np.linalg.norm(vector_a) * np.linalg.norm(vector_b)
    return float(dot / sizes)


print("\nHow similar is each phrase to the others?")
for i in range(len(phrases)):
    for j in range(len(phrases)):
        score = similarity(vectors[i], vectors[j])
        print("   phrase", i, "vs phrase", j, "->", round(score, 2))
print("\nPhrases 0 and 1 (both 'injection') score highest; phrase 2 (crypto) stands apart.")

C:\Users\huber\Documents\CyberAI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4081.14it/s]

Each phrase became a list of 384 numbers.
First phrase, first 8 numbers: [ 0.007 -0.011 -0.062  0.021 -0.116  0.017  0.137 -0.046]

How similar is each phrase to the others?
   phrase 0 vs phrase 0 -> 1.0
   phrase 0 vs phrase 1 -> 0.34
   phrase 0 vs phrase 2 -> -0.01
   phrase 1 vs phrase 0 -> 0.34
   phrase 1 vs phrase 1 -> 1.0
   phrase 1 vs phrase 2 -> 0.02
   phrase 2 vs phrase 0 -> -0.01
   phrase 2 vs phrase 1 -> 0.02
   phrase 2 vs phrase 2 -> 1.0

Phrases 0 and 1 (both 'injection') score highest; phrase 2 (crypto) stands apart.


Optional — the same idea with **OpenAI** embeddings (needs `OPENAI_API_KEY`):

In [4]:
import os
if os.getenv("OPENAI_API_KEY"):
    from openai import OpenAI
    r = OpenAI().embeddings.create(input="SQL injection in the login form",
                                   model="text-embedding-3-small")
    print("OpenAI text-embedding-3-small -> ", len(r.data[0].embedding), "dimensions")
else:
    print("No OPENAI_API_KEY — skipping. (OpenAI small = 1536 dims, large = 3072.)")

No OPENAI_API_KEY — skipping. (OpenAI small = 1536 dims, large = 3072.)


# 📥 3 — Real Cybersecurity Embeddings (1,000 CVEs)

- Omar's dataset: 1,000 CVEs with precomputed vectors + metadata
- `CVE_vectors_1000.tsv` (vectors) + `CVE_metadata_1000.tsv` (details)
- Row *i* of vectors ↔ row *i* of metadata
- Find similar CVEs = **nearest neighbors** in vector space
- This pairing (vector + metadata) is what a vector DB stores

> ### 🎤 Instructor Script
>
> Now real data. We load Omar's dataset of a thousand CVEs: one file of embedding vectors and one of metadata — severity, vendor, the CWE weakness type, a description — lined up row for row. To find vulnerabilities similar to a given one, we normalize the vectors and take a dot product, which gives cosine similarity, then read off the nearest neighbors. Look at the output: the most similar CVEs to our query share its weakness type and impact, even when worded differently. That pairing of a vector with its metadata is exactly what a vector database is built to store and search, which is where we're headed.

In [5]:
import pandas as pd

# load the data: the vectors, and the human-readable details
cve_vectors = np.loadtxt("CVE_vectors_1000.tsv", delimiter="\t")
cve_info = pd.read_csv("CVE_metadata_1000.tsv", sep="\t")
print("Loaded", len(cve_vectors), "CVE vectors and", len(cve_info), "rows of details.\n")
print(cve_info[["CVE_ID", "Severity", "CWE", "Description"]].head(3).to_string(index=False))

# pick one CVE, then find the ones most similar to it
target = 0
print("\nFinding CVEs similar to:", cve_info.loc[target, "CVE_ID"], "-", cve_info.loc[target, "Description"])

# compare the target CVE to every CVE, and remember each score
scored = []
for row in range(len(cve_vectors)):
    score = similarity(cve_vectors[target], cve_vectors[row])
    scored.append((score, row))

scored.sort(reverse=True)            # put the highest scores first
print("\nMost similar CVEs:")
for score, row in scored[1:6]:       # skip position 0 (that is the CVE itself)
    print("  ", round(score, 3), cve_info.loc[row, "CVE_ID"], "-", cve_info.loc[row, "Description"][:50])

Loaded 1000 CVE vectors and 1000 rows of details.

       CVE_ID Severity                       CWE                                                    Description
CVE-2026-0001 Critical              CWE-79 (XSS) Remote Code Execution via CWE-79 in QuantumOS service endpoint
CVE-2027-0001     High             CWE-89 (SQLi)  Privilege escalation due to improper access checks in NovaCMS
CVE-2028-0001     High CWE-120 (Buffer Overflow)        Sensitive data exposure through CWE-120 in SkyDrive API

Finding CVEs similar to: CVE-2026-0001 - Remote Code Execution via CWE-79 in QuantumOS service endpoint

Most similar CVEs:
   0.999 CVE-2027-0038 - Tampering of configuration/state due to CWE-451 we
   0.999 CVE-2026-0049 - Remote Code Execution via CWE-77 in QuantumOS serv
   0.999 CVE-2027-0133 - Privilege escalation due to improper access checks
   0.999 CVE-2026-0087 - Denial of Service via resource exhaustion in Neuro
   0.998 CVE-2026-0027 - Denial of Service via resource exhaustion in He

# ✂️ 4 — Chunking Strategies

- Long documents are split into **chunks** before embedding
- Each chunk becomes one searchable vector
- **Chunk size** & **overlap** are the key knobs
- Too big → answers get diluted; too small → context is lost
- Overlap keeps ideas that straddle a boundary from being cut

> ### 🎤 Instructor Script
>
> You don't embed a whole document as one vector — you split it into chunks, and each chunk becomes one searchable unit. The two knobs are chunk size and overlap. If chunks are too large, a single vector blurs many topics and retrieval gets vague; too small, and you lose the surrounding context that makes a passage meaningful. Overlap means consecutive chunks share some text, so an idea sitting on a boundary isn't sliced in half. We use a simple character splitter on Omar's SSRF document and print the chunk count and a sample. In the lab you'll tune these numbers and watch retrieval quality change.

In [6]:
# read the SSRF document and cut it into small, overlapping pieces ("chunks")
document = open("ssrf.txt", encoding="utf-8").read()

chunk_size = 600     # how many characters go in each chunk
overlap = 100        # how many characters each chunk shares with the one before it

chunks = []
start = 0
while start < len(document):
    piece = document[start : start + chunk_size].strip()
    if piece != "":
        chunks.append(piece)
    start = start + chunk_size - overlap

print("The document is", len(document), "characters long.")
print("We split it into", len(chunks), "chunks.\n")
print("The first chunk looks like this:\n", chunks[0][:300], "...")

The document is 5164 characters long.
We split it into 11 chunks.

The first chunk looks like this:
 Server-Side Request Forgery Prevention Cheat Sheet
Introduction
The objective of the cheat sheet is to provide advice regarding the protection against Server-Side Request Forgery (SSRF) attacks.

This cheat sheet will focus on the defensive point of view and will not explain how to perform this atta ...


# 🗄️ 5 — Vector Database: Install & Operate ChromaDB

- A **vector database** stores embeddings + text + metadata, and searches by similarity fast
- `pip install chromadb` — Omar uses Chroma in the course
- **Create** a collection → **add** chunks (with embeddings) → **query** by meaning
- It handles the indexing and nearest-neighbor search for you
- Add **metadata** to filter (e.g., only High-severity)

> ### 🎤 Instructor Script
>
> Doing similarity by hand is fine for a thousand rows, but real systems use a vector database, which stores your vectors, the original text, and metadata, and runs fast similarity search at scale. We use Chroma — the same database Omar uses — installed with a single pip command. The workflow is three calls: create a collection, add your chunks along with their embeddings and any metadata, and query with an embedded question to get the most similar chunks back. We embed the SSRF chunks with our local model, load them into Chroma, and ask 'What is SSRF?' — watch it return the relevant passages with similarity distances. Then we show a metadata filter, which is how you'd scope a search to, say, only critical findings.

In [7]:
import chromadb

# create a vector database that lives in memory
client = chromadb.EphemeralClient()
collection = client.get_or_create_collection("ssrf_kb")

# turn each chunk into a vector
chunk_vectors = model.encode(chunks)

# build a simple id and a "source" label for each chunk
ids = []
labels = []
for i in range(len(chunks)):
    ids.append("chunk-" + str(i))
    labels.append({"source": "ssrf.txt"})

# add everything (text + vectors + labels) to the database
collection.add(ids=ids,
               documents=chunks,
               embeddings=chunk_vectors.tolist(),
               metadatas=labels)
print("The vector database now holds", collection.count(), "chunks.\n")

# ask a question: turn it into a vector, then find the closest chunks
question = "What is SSRF and how is it exploited?"
question_vector = model.encode(question).tolist()
results = collection.query(query_embeddings=[question_vector], n_results=3)

found_chunks = results["documents"][0]
distances = results["distances"][0]
print("Question:", question)
print("\nClosest chunks in the database (smaller distance = closer in meaning):")
for k in range(len(found_chunks)):
    print("  match", k + 1, "(distance", round(distances[k], 2), "):", found_chunks[k][:80].strip(), "...")

The vector database now holds 11 chunks.

Question: What is SSRF and how is it exploited?

Closest chunks in the database (smaller distance = closer in meaning):
  match 1 (distance 0.7 ): ed to perform an SSRF attack. Take a look at the XXE cheat sheet to learn how to ...
  match 2 (distance 0.78 ): ed, and if poorly handled, can perform specific injection attacks.
Overview of a ...
  match 3 (distance 0.82 ): teract with the internal/external network or the machine itself. One of the enab ...


Persist to disk instead of memory, and filter by metadata — two small changes:

In [8]:
# You can also FILTER by the labels you stored. Here: only chunks whose source is "ssrf.txt".
filtered = collection.query(query_embeddings=[question_vector], n_results=2,
                            where={"source": "ssrf.txt"})
print("Filtered search returned", len(filtered["documents"][0]), "chunks.")

# To SAVE the database to disk (so it survives a restart) instead of memory, use:
#     client = chromadb.PersistentClient(path="my_database_folder")

Filtered search returned 2 chunks.


# ⚡ 6 — Indexing: Why Vector DBs Are Fast

- Comparing a query to *every* vector is **brute force** (slow at scale)
- Vector DBs build an **index** for **approximate** nearest-neighbor search
- Common method: **HNSW** (a navigable graph of vectors)
- Trades a tiny bit of accuracy for huge speed on millions of vectors
- Chroma builds and manages this index for you

> ### 🎤 Instructor Script
>
> One more concept the course covers: indexing. When you only have a thousand vectors, comparing a query against all of them — brute force — is instant. But at millions or billions of vectors that's too slow, so vector databases build an index for approximate nearest-neighbor search. The popular method, HNSW, organizes vectors into a navigable graph so a query can hop to its neighborhood in a few steps instead of scanning everything. You trade a sliver of accuracy for an enormous speed-up. The good news is Chroma builds and maintains this index automatically — you just add and query — but it's worth knowing what's happening under the hood when you scale up.

In [9]:
import time

# "by hand" search: compare the question to ALL 1,000 CVE vectors, and time it
start_time = time.time()
hand_scores = []
for row in range(len(cve_vectors)):
    hand_scores.append(similarity(cve_vectors[0], cve_vectors[row]))
elapsed_ms = (time.time() - start_time) * 1000

print("Comparing against", len(cve_vectors), "vectors took", round(elapsed_ms, 1), "ms.")
print("That is fast for 1,000 vectors. For millions, a vector database uses an index (HNSW)")
print("to stay fast — and ChromaDB builds that index for you, so you never write this loop.")

Comparing against 1000 vectors took 11.5 ms.
That is fast for 1,000 vectors. For millions, a vector database uses an index (HNSW)
to stay fast — and ChromaDB builds that index for you, so you never write this loop.


# 📊 7 — See the Space (Visualization)

- Project high-dim vectors to 2-D with **PCA** to eyeball structure
- Similar CVEs cluster; color by severity or CWE to see patterns
- Interactive: the **TensorFlow Embedding Projector** (browser)
- Upload the two TSVs and explore PCA / t-SNE / UMAP

> ### 🎤 Instructor Script
>
> Finally, let's see the space. We can't picture hundreds of dimensions, so we use PCA to flatten the CVE vectors to two dimensions and color them by severity — clusters mean the embedding captured real structure. If matplotlib is blocked on your machine, the cell prints a note and you can use the interactive TensorFlow Embedding Projector instead: just upload the two TSV files at projector.tensorflow.org and fly through the data in 3-D, switching between PCA, t-SNE, and UMAP and clicking any CVE to highlight its neighbors. Either way, the goal is the same — build intuition that meaning really is geometry.

In [10]:
from sklearn.decomposition import PCA

# squeeze each CVE vector down to just 2 numbers so we can plot it on a flat chart
flat = PCA(n_components=2).fit_transform(cve_vectors)
colors = {"Critical": "red", "High": "orange", "Medium": "blue", "Low": "green"}

try:
    if plt is None:
        raise RuntimeError("matplotlib not available")
    plt.figure(figsize=(7, 5))
    for level in colors:
        # collect the (x, y) points for every CVE of this severity
        x_points = []
        y_points = []
        for row in range(len(cve_info)):
            if cve_info.loc[row, "Severity"] == level:
                x_points.append(flat[row, 0])
                y_points.append(flat[row, 1])
        plt.scatter(x_points, y_points, s=10, alpha=0.6, label=level, color=colors[level])
    plt.legend(title="Severity")
    plt.title("CVE embeddings squeezed to 2-D (similar CVEs sit close together)")
    plt.tight_layout(); plt.show()
except Exception as e:
    print("Plot skipped:", e)
    print("Open this notebook on Google Colab to see the chart, or upload the two TSV files")
    print("to https://projector.tensorflow.org for an interactive view.")

Plot skipped: DLL load failed while importing _backend_agg: An Application Control policy has blocked this file.
Open this notebook on Google Colab to see the chart, or upload the two TSV files
to https://projector.tensorflow.org for an interactive view.


# ✅ Summary — You Built the Retrieval Engine

- Embeddings turn text into **meaning-vectors** (same model for all text)
- **Cosine similarity** → nearest-neighbor / semantic search
- **Chunking** prepares documents; size & overlap matter
- A **vector database** (ChromaDB) stores + indexes + searches vectors
- Next: `rag_example.ipynb` adds **augment + generate** = full RAG

> ### 🎤 Instructor Script
>
> Here's what you built, in Omar's order: you saw what embeddings are and that one model must embed everything, used cosine similarity to find nearest neighbors on real CVE data, chunked a document for ingestion, and stood up an actual vector database with Chroma — create, add, query, filter — plus the indexing idea that makes it scale. That is the entire retrieval half of RAG. In the companion notebook we add the last two steps, augmentation and generation, to turn this engine into a question-answering system grounded in your documents — still with no framework. The framework version, LangChain, is the next module.

In [11]:
print("You can now:")
print("  - explain embeddings, dimensionality, and semantic search (no TF-IDF needed)")
print("  - chunk documents and reason about size/overlap")
print("  - install and operate a vector database (ChromaDB): create, add, query, filter")
print("  - explain why indexing (HNSW) makes vector search scale")
print("\nNext: rag_example.ipynb -> retrieve (Chroma) + augment + generate = a full RAG bot.")

You can now:
  - explain embeddings, dimensionality, and semantic search (no TF-IDF needed)
  - chunk documents and reason about size/overlap
  - install and operate a vector database (ChromaDB): create, add, query, filter
  - explain why indexing (HNSW) makes vector search scale

Next: rag_example.ipynb -> retrieve (Chroma) + augment + generate = a full RAG bot.
